# Module 15 Lab — Anthropic Python SDK: Production Research Assistant

In this lab you will build a **research assistant** using the Anthropic SDK directly — no framework abstractions. By the end you will have covered:

- The API contract: content blocks, `stop_reason`, and message structure
- The full **tool-use loop** (tool_use → tool_result → continue)
- **Streaming** with the context-manager API and token accounting
- **Prompt caching** with `cache_control` and cache-hit metrics
- **Production patterns**: tenacity retries and a cost tracker

Work through each section in order. Every TODO has a hidden solution — reveal it only if you are stuck.

In [ ]:
!pip install anthropic tenacity

In [ ]:
import os
import anthropic

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
if not ANTHROPIC_API_KEY:
    raise ValueError("Set ANTHROPIC_API_KEY in Colab Secrets or environment")

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
MODEL = "claude-haiku-4-5-20251001"
print("Client ready — model:", MODEL)

---
## Section 1: The API Contract

Every response from `messages.create()` contains:

| Field | What it means |
|---|---|
| `content` | List of **content blocks** — `TextBlock`, `ToolUseBlock`, etc. |
| `stop_reason` | Why generation stopped: `end_turn`, `tool_use`, `max_tokens`, `stop_sequence` |
| `usage` | `input_tokens` and `output_tokens` for this call |

A `TextBlock` has `.type == "text"` and `.text`.  
A `ToolUseBlock` has `.type == "tool_use"`, `.id`, `.name`, and `.input` (a dict).

Run the cell below to see a real response structure.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=256,
    messages=[
        {"role": "user", "content": "What is the capital of France? Answer in one sentence."}
    ]
)

print("stop_reason:", response.stop_reason)
print("usage:", response.usage)
print("content blocks:", response.content)
print("\nText:", response.content[0].text)

---
## Section 2: Tool Use Loop

When the model wants to call a tool, `stop_reason` is `"tool_use"` and `content` contains a `ToolUseBlock`. Your code must:

1. Detect `stop_reason == "tool_use"`
2. Extract all `ToolUseBlock` items from `content`
3. Execute each tool
4. Append the assistant message, then a user message containing `tool_result` blocks
5. Call `messages.create()` again — repeat until `stop_reason == "end_turn"`

The `tool_result` content block looks like:
```python
{"type": "tool_result", "tool_use_id": "<id from ToolUseBlock>", "content": "<result string>"}
```

Complete the four TODOs below to implement a working agent loop.

In [ ]:
# Stub tool — replace with a real web search in production
def search_web(query: str) -> str:
    """Simulates a web search. Returns a short paragraph of results."""
    return (
        f"[Simulated search results for '{query}']: "
        "According to recent sources, this topic involves several key concepts. "
        "Researchers have found that the primary factors include efficiency, scalability, "
        "and reliability. Multiple studies confirm these findings across different contexts."
    )

TOOL_REGISTRY = {"search_web": search_web}

### TODO 1 — Define the tools list

Build the `tools` list that gets passed to `messages.create(tools=...)`.  
Each tool is a dict with `name`, `description`, and `input_schema` (JSON Schema).

The `search_web` tool takes one string argument: `query`.

In [ ]:
# TODO 1: Define the tools list for messages.create()
# Each entry needs: name, description, input_schema (JSON Schema object)

tools = [
    # YOUR CODE HERE
]

# === SOLUTION (reveal only if stuck) ===
# tools = [
#     {
#         "name": "search_web",
#         "description": "Search the web for current information on a topic. Returns a short summary of results.",
#         "input_schema": {
#             "type": "object",
#             "properties": {
#                 "query": {
#                     "type": "string",
#                     "description": "The search query to look up"
#                 }
#             },
#             "required": ["query"]
#         }
#     }
# ]

### TODO 2, 3, 4 — Implement the agent loop

Fill in the three marked sections inside `run_agent()`.

In [ ]:
def run_agent(question: str, tools: list, max_turns: int = 5) -> str:
    """Run a tool-use agent loop until end_turn or max_turns reached."""
    messages = [{"role": "user", "content": question}]

    for turn in range(max_turns):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=tools,
            messages=messages
        )
        print(f"Turn {turn + 1}: stop_reason={response.stop_reason}")

        if response.stop_reason == "end_turn":
            # Extract the final text answer
            for block in response.content:
                if block.type == "text":
                    return block.text
            return "(no text in final response)"

        # TODO 2: Detect tool_use blocks and extract them
        # Hint: iterate response.content and filter by block.type == "tool_use"
        tool_use_blocks = []  # YOUR CODE HERE

        if not tool_use_blocks:
            break

        # Append the assistant turn to the conversation
        messages.append({"role": "assistant", "content": response.content})

        # TODO 3: Execute each tool and build tool_result content blocks
        # Hint: call TOOL_REGISTRY[block.name](**block.input) for each tool_use block
        # Build a list of {"type": "tool_result", "tool_use_id": ..., "content": ...}
        tool_results = []  # YOUR CODE HERE

        # TODO 4: Append the user message containing all tool results and continue
        # Hint: messages.append({"role": "user", "content": tool_results})
        # YOUR CODE HERE

    return "Max turns reached without final answer"


# === SOLUTION (reveal only if stuck) ===
# def run_agent(question: str, tools: list, max_turns: int = 5) -> str:
#     messages = [{"role": "user", "content": question}]
#     for turn in range(max_turns):
#         response = client.messages.create(
#             model=MODEL, max_tokens=1024, tools=tools, messages=messages
#         )
#         print(f"Turn {turn + 1}: stop_reason={response.stop_reason}")
#         if response.stop_reason == "end_turn":
#             for block in response.content:
#                 if block.type == "text":
#                     return block.text
#             return "(no text)"
#         # TODO 2 solution
#         tool_use_blocks = [b for b in response.content if b.type == "tool_use"]
#         if not tool_use_blocks:
#             break
#         messages.append({"role": "assistant", "content": response.content})
#         # TODO 3 solution
#         tool_results = []
#         for block in tool_use_blocks:
#             print(f"  Calling tool: {block.name}({block.input})")
#             result = TOOL_REGISTRY[block.name](**block.input)
#             tool_results.append({
#                 "type": "tool_result",
#                 "tool_use_id": block.id,
#                 "content": result
#             })
#         # TODO 4 solution
#         messages.append({"role": "user", "content": tool_results})
#     return "Max turns reached"


# Test it once your TODOs are filled in
answer = run_agent(
    "What are the main benefits of retrieval-augmented generation? Search the web and summarise.",
    tools=tools
)
print("\nFinal answer:\n", answer)

---
## Section 3: Streaming

Use `client.messages.stream()` as a context manager to print tokens as they arrive. The `.text_stream` iterator yields each text chunk as it comes.

After the stream closes, call `.get_final_message()` to get the full `Message` object with usage stats.

In [ ]:
print("Streaming response:\n")
with client.messages.stream(
    model=MODEL,
    max_tokens=512,
    messages=[
        {"role": "user", "content": "Explain prompt caching in three short bullet points."}
    ]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

print("\n\n--- Stream finished ---")

### TODO — Token accounting after streaming

Call `stream.get_final_message()` **inside** the `with` block (before it closes) to retrieve the final message, then print `usage.input_tokens` and `usage.output_tokens`.

In [ ]:
# TODO: Add token counting via get_final_message()
# Modify this cell to capture and print usage after streaming

with client.messages.stream(
    model=MODEL,
    max_tokens=256,
    messages=[{"role": "user", "content": "What is LangGraph in one sentence?"}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)
    # YOUR CODE HERE: get final_message and print its usage

# === SOLUTION (reveal only if stuck) ===
# with client.messages.stream(
#     model=MODEL,
#     max_tokens=256,
#     messages=[{"role": "user", "content": "What is LangGraph in one sentence?"}]
# ) as stream:
#     for text in stream.text_stream:
#         print(text, end="", flush=True)
#     final_msg = stream.get_final_message()
#     usage = final_msg.usage
#     print(f"\n\nTokens — input: {usage.input_tokens}, output: {usage.output_tokens}")

---
## Section 4: Prompt Caching

Prompt caching lets you mark parts of a prompt with `"cache_control": {"type": "ephemeral"}`. The first call computes and stores that prefix; subsequent calls within 5 minutes read from cache instead.

Cache hits appear in `usage` as `cache_read_input_tokens` — those tokens cost about 10% of the normal input rate.  
Cache writes appear as `cache_creation_input_tokens`.

To enable: add `"cache_control": {"type": "ephemeral"}` to the **last content block** of the prompt section you want cached.

In [ ]:
# A long system prompt (simulate a large knowledge base / reference document)
LONG_SYSTEM_PROMPT = (
    "You are an expert AI systems researcher. "
    + ("You have deep knowledge of transformer architectures, attention mechanisms, "
       "reinforcement learning from human feedback, constitutional AI, chain-of-thought "
       "prompting, retrieval-augmented generation, tool use, multi-agent orchestration, "
       "and agentic evaluation frameworks. ") * 20  # repeat to exceed 1024-token cache threshold
    + "Answer all questions concisely and accurately."
)

def ask_with_cache(question: str):
    response = client.messages.create(
        model=MODEL,
        max_tokens=256,
        system=[
            {
                "type": "text",
                "text": LONG_SYSTEM_PROMPT,
                "cache_control": {"type": "ephemeral"}  # mark this block for caching
            }
        ],
        messages=[{"role": "user", "content": question}]
    )
    return response

print("First call (expect cache_creation_input_tokens > 0):")
r1 = ask_with_cache("What is constitutional AI?")
print("  usage:", r1.usage)

print("\nSecond call (expect cache_read_input_tokens > 0):")
r2 = ask_with_cache("What is RLHF?")
print("  usage:", r2.usage)

### TODO — Compare cache savings

Print the `cache_read_input_tokens` from both calls side by side. Calculate how many tokens were saved on the second call compared to a non-cached call (hint: `cache_read_input_tokens` on call 2 vs `input_tokens` on call 1).

In [ ]:
# TODO: Compare cache_read_input_tokens between r1 and r2
# Print which call got a cache hit and how many tokens were served from cache

# YOUR CODE HERE

# === SOLUTION (reveal only if stuck) ===
# r1_cached = getattr(r1.usage, "cache_read_input_tokens", 0) or 0
# r2_cached = getattr(r2.usage, "cache_read_input_tokens", 0) or 0
# r1_created = getattr(r1.usage, "cache_creation_input_tokens", 0) or 0
#
# print(f"Call 1 — cache_creation: {r1_created}, cache_read: {r1_cached}")
# print(f"Call 2 — cache_creation: {getattr(r2.usage, 'cache_creation_input_tokens', 0)}, cache_read: {r2_cached}")
#
# if r2_cached > 0:
#     print(f"\nCache hit on call 2! {r2_cached} tokens served from cache.")
#     print(f"Without cache, those {r2_cached} tokens would have been billed at full input rate.")

---
## Section 5: Production — Retries and Cost Tracking

Two production essentials:

1. **tenacity retries** — wraps `messages.create()` with exponential backoff for transient `APIStatusError` (rate limits, 529 overload).
2. **CostTracker** — accumulates cost per call using published Haiku 3.5 pricing:
   - Input: `$0.25 / 1M tokens` → `$0.00000025` per token
   - Output: `$1.25 / 1M tokens` → `$0.00000125` per token

The retry decorator below is ready to use. Fill in the `CostTracker` TODO.

In [ ]:
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

@retry(
    wait=wait_exponential(multiplier=1, min=1, max=60),
    stop=stop_after_attempt(3),
    retry=retry_if_exception_type(anthropic.APIStatusError),
    reraise=True
)
def resilient_create(**kwargs):
    """messages.create() with automatic exponential-backoff retries."""
    return client.messages.create(**kwargs)


# Quick smoke test
r = resilient_create(
    model=MODEL,
    max_tokens=64,
    messages=[{"role": "user", "content": "Say 'retry works' and nothing else."}]
)
print(r.content[0].text)

### TODO — Build a CostTracker

Implement a `CostTracker` class with:
- `record(usage)` — takes an `anthropic.types.Usage` object, accumulates total cost
- `total_cost` property — returns accumulated USD cost
- `summary()` — prints total tokens and total cost

Use these rates:
- Input: `0.00000025` USD per token
- Output: `0.00000125` USD per token

In [ ]:
# TODO: Implement CostTracker
class CostTracker:
    INPUT_RATE = 0.00000025   # USD per input token (claude-haiku-4-5)
    OUTPUT_RATE = 0.00000125  # USD per output token

    def __init__(self):
        # YOUR CODE HERE — initialise counters
        pass

    def record(self, usage):
        """Accumulate cost from a Usage object."""
        # YOUR CODE HERE
        pass

    @property
    def total_cost(self) -> float:
        # YOUR CODE HERE
        return 0.0

    def summary(self):
        # YOUR CODE HERE — print total_input_tokens, total_output_tokens, total_cost
        pass


# Test with a few calls
tracker = CostTracker()

for question in [
    "What is a neural network?",
    "What is attention in transformers?",
    "What is fine-tuning?"
]:
    resp = resilient_create(
        model=MODEL,
        max_tokens=128,
        messages=[{"role": "user", "content": question}]
    )
    tracker.record(resp.usage)

tracker.summary()


# === SOLUTION (reveal only if stuck) ===
# class CostTracker:
#     INPUT_RATE = 0.00000025
#     OUTPUT_RATE = 0.00000125
#
#     def __init__(self):
#         self._input_tokens = 0
#         self._output_tokens = 0
#
#     def record(self, usage):
#         self._input_tokens += usage.input_tokens
#         self._output_tokens += usage.output_tokens
#
#     @property
#     def total_cost(self) -> float:
#         return (self._input_tokens * self.INPUT_RATE +
#                 self._output_tokens * self.OUTPUT_RATE)
#
#     def summary(self):
#         print(f"Total input tokens : {self._input_tokens}")
#         print(f"Total output tokens: {self._output_tokens}")
#         print(f"Estimated cost     : ${self.total_cost:.6f} USD")

---
## Wrap-up

You have built a production-ready research assistant using the Anthropic SDK directly. Key takeaways:

- **Tool use** is a request–response loop: detect `stop_reason == "tool_use"`, execute, feed `tool_result` back.
- **Streaming** prints tokens live; `get_final_message()` gives you usage stats.
- **Prompt caching** can eliminate redundant re-computation of large static prompts — check `cache_read_input_tokens`.
- **tenacity** handles transient API errors with minimal boilerplate.
- **CostTracker** closes the loop on observability — always know what you're spending.

Next: Module 16 — LangGraph for stateful, graph-based agent workflows.